# Inspect pipeline.py

Notebook di ispezione: stessa sequenza di chiamate di `pipeline.py` (funzioni `run_cleaning1`, `run_cleaning2`, `run_all`), ma una cella per passaggio cosi' si puo' controllare il DataFrame dopo ogni step.

Nessuna logica e' stata modificata: le celle chiamano le funzioni di `pipeline.py` esattamente nell'ordine in cui compaiono negli orchestratori.

Dataset di riferimento: `config.ADNIMERGE` (cambia `cfg = ...` per ispezionare un altro dataset, es. `PTDEMOG`, `PLASMA_PANEL`, `PLASMA_NFL`).

## Setup

In [ ]:
import numpy as np
import pandas as pd

import config
from config import DatasetConfig, ADNIMERGE, PTDEMOG
from pipeline import *

cfg = ADNIMERGE

## CLEANING 1 — funzioni atomiche (df -> df)

Segue l'ordine di `run_cleaning1()` in pipeline.py.

### load — download_file (ex client.download_file)

In [ ]:
df = load(cfg)
df.shape

In [ ]:
df.head()

### drop_columns — rimuove colonne indesiderate (es. VISCODE in PTDEMOG)

In [ ]:
df, dropped_cols, missing_drop_cols = drop_columns(df, cfg.drop_columns)
print("scartate:", dropped_cols)
print("richieste ma assenti:", missing_drop_cols)
df.shape

### replace_unknown — replace_unknown_values

In [ ]:
df = replace_unknown(df)
df.head()

### decensor — toglie i simboli di censura (solo se cfg.decensor_biomarkers)

In [ ]:
if cfg.decensor_biomarkers:
    df = decensor(df, cfg.decensor_columns or config.columns_in("Biomarker"))
df.shape

### drop_if_all_none — 1° drop: colonne essenziali

In [ ]:
n_before = len(df)
df = drop_if_all_none(df, cfg.essential_columns)
print("righe rimosse:", n_before - len(df))
df.shape

### drop_if_all_none — 2° drop: DX obbligatoria (also_required)

In [ ]:
n_before = len(df)
df = drop_if_all_none(df, cfg.also_required)
print("righe rimosse:", n_before - len(df))
df.shape

### rename_variables — new_variable_names (CATALOG + override per-file)

In [ ]:
df = rename_variables(df, cfg)
df.columns.tolist()

### parse_dates — to_date_format

In [ ]:
df = parse_dates(df, [cfg.date_column] + cfg.extra_date_columns)
df.dtypes

### dedup_visits — find_exam_code (elimina visite duplicate, tiene la piu' completa)

In [ ]:
n_before = len(df)
df = dedup_visits(df, cfg.id_column, cfg.date_column, cfg.essential_columns)
print("righe rimosse:", n_before - len(df))
df.shape

### add_visit_month — find_exam_code -> VISIT_MONTH

In [ ]:
df = add_visit_month(df, cfg.id_column, cfg.date_column)
df[[cfg.id_column, cfg.date_column, "VISIT_MONTH"]].head()

### recompute_age — add_calculated_age (solo se cfg.recompute_age)

In [ ]:
if cfg.recompute_age:
    df = recompute_age(df, cfg.date_column)
df.filter(regex="AGE").head()

### recode — categorize_* (stringa -> codice numerico, config.RECODE)

In [ ]:
df = recode(df, cfg.recode_columns)
df[[c for c in cfg.recode_columns if c in df.columns]].head()

### clean_fs_fields — FLDSTRENG/FSVERSION .str.extract (solo se cfg.clean_fs_fields)

In [ ]:
if cfg.clean_fs_fields:
    df = clean_fs_fields(df)
df.filter(regex="FLDSTRENG|FSVERSION").head()

### add_constant_columns — stampa colonne-costante (armonizzazione, es. METHOD_PLASMA)

In [ ]:
df = add_constant_columns(df, cfg.constant_columns)
df.shape

### add_derived_ratios — rapporti generici (es. AB42/AB40)

In [ ]:
df = add_derived_ratios(df, cfg.derived_ratios)
df.shape

### add_atn_profile — get_ATN_profile (solo se cfg.compute_atn; ADNIMERGE non lo usa)

In [ ]:
if cfg.compute_atn:
    df = add_atn_profile(df, cfg)
df.shape

### df1 — risultato di cleaning 1 (equivalente a run_cleaning1(cfg))

In [ ]:
df1 = df
df1.shape

In [ ]:
df1.head()

## CLEANING 2 / 3 — ex notebook adni_cleaning2 / adni_cleaning3

Segue l'ordine di `run_cleaning2(df, cfg)` in pipeline.py. Si riparte da `df1`.

In [ ]:
df = df1.copy()
log = {}
r0, c0 = df.shape

### drop_sparse_columns — remove_param_few_subjects (solo se cfg.drop_sparse_columns)

In [ ]:
if cfg.drop_sparse_columns:
    df, dropped = drop_sparse_columns(df)
    log["colonne_scartate_sparse"] = dropped
df.shape

### remove_single_visit_subjects — remove_sub_1visit (solo se cfg.remove_single_visit)

In [ ]:
if cfg.remove_single_visit:
    n_before = len(df)
    df, info = remove_single_visit_subjects(df, cfg.id_column, force=cfg.keep_even_single_visit)
    log["single_visit"] = {**info, "righe_rimosse": n_before - len(df)}
df.shape

### make_dummies — classes_to_dummies (solo se cfg.make_dummies)

In [ ]:
created = []
if cfg.make_dummies:
    df, created = make_dummies(df, cfg.dummy_columns)
    log["dummy_create"] = created
print("dummy create:", created)
df.shape

### drop_if_all_none — drop_if_all_none (volumi, riuso di cleaning 1)

In [ ]:
if cfg.volume_row_keys:
    n_before = len(df)
    df = drop_if_all_none(df, cfg.volume_row_keys)
    log["righe_rimosse_volumi"] = n_before - len(df)
df.shape

### normalize_volumes_icv — transform_volumes_as_ICV_percent (solo se cfg.normalize_icv)

In [ ]:
if cfg.normalize_icv:
    df = normalize_volumes_icv(df, cfg.icv_column)
    log["icv_normalizzato"] = True
df.shape

### keep_only_columns — filtro colonne finali (whitelist)

In [ ]:
if cfg.keep_columns:
    engineered = [c for c in ("VISIT_MONTH", "AGE_bl") if c in df.columns]
    df, dropped_cols, missing_cols = keep_only_columns(df, cfg.keep_columns, extra=created + engineered)
    log["colonne_scartate_finali"] = dropped_cols
    log["colonne_richieste_assenti"] = missing_cols
df.shape

### df2, log — risultato di cleaning 2 (equivalente a run_cleaning2(df1, cfg))

In [ ]:
log["shape_iniziale"] = (r0, c0)
log["shape_finale"] = df.shape
df2 = df
log

In [ ]:
df2.head()

## MERGE per categoria (v0)

Motore generico guidato da `config.CATEGORY_MERGE`. Esempio: la categoria `plasma` ha 2 file registrati (`PLASMA_PANEL`, `PLASMA_NFL`) — servono entrambi puliti (cleaning 1 + 2) prima del merge.

In [ ]:
from config import PLASMA_PANEL, PLASMA_NFL

In [ ]:
# run_cleaning(cfg) restituisce (df, log): prendiamo il df pulito (cleaning1 + cleaning2)
datasets = {
    PLASMA_PANEL.file_code: run_cleaning(PLASMA_PANEL)[0],
    PLASMA_NFL.file_code: run_cleaning(PLASMA_NFL)[0],
}
merged, merge_log = merge_category(datasets, "plasma")
merge_log

In [ ]:
merged.head()

## REPORT

Rigenera cio' che prima era l'Excel `_statistics` (output, non input). Report su `df1` (cleaned 1), come in `run_all()`.

In [ ]:
report = profile(df1, cfg.cohort_column)
report.head(20)

## SAVE

Unico punto (insieme al blocco `__main__`) che tocca il disco. Celle NON eseguite di default: eseguile solo se vuoi davvero sovrascrivere i CSV in `cfg.output_cleaned1` / `cfg.output_cleaned2` / `cfg.report_file`.

In [ ]:
# save_dataset(df1, cfg.output_cleaned1)
# report.to_csv(cfg.report_file, index=False)
# save_dataset(df2, cfg.output_cleaned2)

## run_all — orchestratore completo (tutti i file registrati in config.DATASETS)

Cella NON eseguita di default: `run_all()` pulisce tutti i dataset registrati, mergia le categorie e scrive tutti i CSV su disco (come `python pipeline.py`).

In [ ]:
# summary = run_all()
# summary